# Módulo 5 · Notebook 4 — Introducción a LangGraph

LangGraph permite modelar flujos de agentes como un **grafo de estados**.

En este notebook aprenderás:
- qué es el `state` compartido,
- cómo definir nodos (funciones),
- cómo enrutar con condiciones,
- y cómo ejecutar el grafo con preguntas sobre partidos de los Mundiales (CSV).

## 1. Setup

In [1]:
import warnings
from pathlib import Path
from typing import Literal, TypedDict

import pandas as pd
from langchain_ollama import ChatOllama
from langgraph.graph import END, START, StateGraph

warnings.filterwarnings("ignore")

BASE_DIR = Path("../../").resolve()
CSV_PATH = BASE_DIR / "data" / "table" / "WorldCupMatches.csv"

llm = ChatOllama(model="llama3.2:3b", temperature=0)
print("✅ Setup listo")

/Users/guane/Documentos/GlogalLogic/AI-course/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


✅ Setup listo


## 2. Cargar y preparar datos

In [2]:
df = pd.read_csv(CSV_PATH)

df["Datetime"] = pd.to_datetime(df["Datetime"].str.strip(), errors="coerce", dayfirst=True)
df["Attendance"] = pd.to_numeric(df["Attendance"], errors="coerce")
df["HomeTeamGoals"] = pd.to_numeric(df["HomeTeamGoals"], errors="coerce").fillna(0)
df["AwayTeamGoals"] = pd.to_numeric(df["AwayTeamGoals"], errors="coerce").fillna(0)
df["TotalGoals"] = df["HomeTeamGoals"] + df["AwayTeamGoals"]

print(f"Partidos cargados: {len(df)}")
display(df.head(3))

Partidos cargados: 836


,Year,Datetime,Stage,Stadium,City,HomeTeamName,HomeTeamGoals,AwayTeamGoals,AwayTeamName,WinConditions,...,HalfTimeHomeGoals,HalfTimeAwayGoals,Referee,Assistant1,Assistant2,RoundID,MatchID,HomeTeamInitials,AwayTeamInitials,TotalGoals
0,1930,1930-07-13 15:00:00,Group 1,Pocitos,Montevideo,France,4,1,Mexico,,...,3,0,LOMBARDI Domingo (URU),CRISTOPHE Henry (BEL),REGO Gilberto (BRA),201,1096,FRA,MEX,5
1,1930,1930-07-13 15:00:00,Group 4,Parque Central,Montevideo,USA,3,0,Belgium,,...,2,0,MACIAS Jose (ARG),MATEUCCI Francisco (URU),WARNKEN Alberto (CHI),201,1090,USA,BEL,3
2,1930,1930-07-14 12:45:00,Group 2,Parque Central,Montevideo,Yugoslavia,2,1,Brazil,,...,2,0,TEJADA Anibal (URU),VALLARINO Ricardo (URU),BALWAY Thomas (FRA),201,1093,YUG,BRA,3


## 3. Diseñar el estado del grafo

El `state` es el objeto compartido entre nodos. Cada nodo lee y/o actualiza campos del estado.

In [3]:
class GraphState(TypedDict):
    question: str
    route: Literal["analista", "historiador", "mixto"]
    analysis: str
    narrative: str
    final_answer: str

## 4. Funciones auxiliares de datos

In [4]:
def match_with_max_attendance(year: int | None = None) -> str:
    data = df.copy()
    if year is not None:
        data = data[data["Year"] == year]
    if data.empty:
        return "No hay datos para ese año."

    row = data.sort_values("Attendance", ascending=False).iloc[0]
    return (
        f"Mayor asistencia: {row['HomeTeamName']} {int(row['HomeTeamGoals'])}-{int(row['AwayTeamGoals'])} {row['AwayTeamName']}"
        f" | Año: {int(row['Year'])} | Stage: {row['Stage']} | Attendance: {int(row['Attendance'])}"
    )


def avg_goals_by_stage(stage_keyword: str) -> str:
    mask = df["Stage"].str.contains(stage_keyword, case=False, na=False)
    data = df[mask]
    if data.empty:
        return f"No encontré partidos para stage con '{stage_keyword}'."
    avg_goals = data["TotalGoals"].mean()
    return f"Promedio de goles en stages que contienen '{stage_keyword}': {avg_goals:.2f}"


## 5. Definir nodos del grafo

In [ ]:
def router_node(state: GraphState) -> GraphState:
    q = state["question"].lower()
    has_numeric = any(k in q for k in ["promedio", "cuánt", "cuanto", "asistencia", "dato", "goles"])
    has_story = any(k in q for k in ["historia", "contexto", "explica", "importancia", "por qué", "por que"])

    if has_numeric and has_story:
        route = "mixto"
    elif has_numeric:
        route = "analista"
    else:
        route = "historiador"

    return {**state, "route": route}


def analyst_node(state: GraphState) -> GraphState:
    q = state["question"].lower()

    if "asistencia" in q:
        if "1930" in q:
            analysis = match_with_max_attendance(1930)
        else:
            analysis = match_with_max_attendance()
    elif "final" in q:
        analysis = avg_goals_by_stage("Final")
    elif "semi" in q:
        analysis = avg_goals_by_stage("Semi")
    else:
        analysis = (
            "Resumen rápido: "
            f"{len(df)} partidos en total, promedio global de goles {df['TotalGoals'].mean():.2f}."
        )

    return {**state, "analysis": analysis}


def historian_node(state: GraphState) -> GraphState:
    prompt = f"""
Eres un historiador de los Mundiales de la FIFA.
Pregunta: {state['question']}

Si existe, usa este dato analítico como apoyo: {state.get('analysis', '')}
Responde en español, claro y breve (3-5 líneas).
"""
    narrative = llm.invoke(prompt).content
    return {**state, "narrative": narrative}


def compose_node(state: GraphState) -> GraphState:
    route = state["route"]
    if route == "analista":
        final = f"[Ruta: analista]\n{state.get('analysis', '')}"
    elif route == "historiador":
        final = f"[Ruta: historiador]\n{state.get('narrative', '')}"
    else:
        final = (
            "[Ruta: mixto]\n"
            f"Dato analítico: {state.get('analysis', '')}\n\n"
            f"Contexto histórico: {state.get('narrative', '')}"
        )
    return {**state, "final_answer": final}


## 6. Construir y compilar el StateGraph

In [6]:
builder = StateGraph(GraphState)

builder.add_node("router", router_node)
builder.add_node("analyst", analyst_node)
builder.add_node("historian", historian_node)
builder.add_node("compose", compose_node)

builder.add_edge(START, "router")

def route_from_router(state: GraphState):
    if state["route"] == "analista":
        return "analyst"
    if state["route"] == "historiador":
        return "historian"
    return "analyst"  # mixto: primero análisis

builder.add_conditional_edges(
    "router",
    route_from_router,
    {
        "analyst": "analyst",
        "historian": "historian",
    },
)

def route_after_analyst(state: GraphState):
    return "historian" if state["route"] == "mixto" else "compose"

builder.add_conditional_edges(
    "analyst",
    route_after_analyst,
    {
        "historian": "historian",
        "compose": "compose",
    },
)

builder.add_edge("historian", "compose")
builder.add_edge("compose", END)

graph = builder.compile()
print("✅ Grafo compilado")

✅ Grafo compilado


## 7. Ejecutar el grafo

In [7]:
questions = [
    "¿Cuál fue el partido con mayor asistencia en 1930?",
    "Explica por qué la final de 1930 fue importante.",
    "Explica la final y además dame un dato de asistencia.",
]

for q in questions:
    state_in = {
        "question": q,
        "route": "analista",
        "analysis": "",
        "narrative": "",
        "final_answer": "",
    }
    result = graph.invoke(state_in)
    print("=" * 100)
    print("Pregunta:", q)
    print(result["final_answer"])


Pregunta: ¿Cuál fue el partido con mayor asistencia en 1930?
[Ruta: analista]
Mayor asistencia: Uruguay 6-1 Yugoslavia | Año: 1930 | Stage: Semi-finals | Attendance: 79867


Pregunta: Explica por qué la final de 1930 fue importante.
[Ruta: historiador]
La final del Mundial de la FIFA de 1930 entre Uruguay y Argentina es considerada una de las más importantes en la historia del torneo. Este partido marcó el debut oficial del Mundial de fútbol y se llevó a cabo en Montevideo, Uruguay. La victoria de Uruguay por 4-2 fue un gran éxito para el país, que se convirtió en el primer campeón del mundo en la historia del torneo.
Pregunta: Explica la final y además dame un dato de asistencia.
[Ruta: mixto]
Dato analítico: Mayor asistencia: Uruguay 2-1 Brazil | Año: 1950 | Stage: Group 6 | Attendance: 173850

Contexto histórico: La final del Mundial de la FIFA es el partido más importante del torneo. En general, se juega entre los dos equipos que han avanzado a la última ronda sin perder ni empatar.

En cuanto al dato de asistencia, según mi conocimiento, la final del Mundial de 1950 entre Uruguay y Brasil tuvo una asistencia récord de 173.850 espectadores en el Estadi

## 8. Visualizar el grafo (Mermaid)

Si tu entorno lo soporta, puedes copiar este Mermaid para documentar el flujo.

In [8]:
print(graph.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	router(router)
	analyst(analyst)
	historian(historian)
	compose(compose)
	__end__([<p>__end__</p>]):::last
	__start__ --> router;
	analyst -.-> compose;
	analyst -.-> historian;
	historian --> compose;
	router -.-> analyst;
	router -.-> historian;
	compose --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



## 9. Ejercicios sugeridos

1. Agrega un nodo `validator` que detecte preguntas fuera de tema.
2. Cambia el router para distinguir entre `Final`, `Semi-finals` y `Group`.
3. Incluye una tool o función para responder: "Top 5 partidos con más goles".
4. Persistencia: guarda cada `state` final en un archivo JSON para auditoría.